In [3]:
import requests
import os,json
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
CORPUS_KEY = "RAGBOOK"
VECATARA_API_KEY = os.getenv("VECTARA_API_KEY")

In [5]:
url = f"https://api.vectara.io/v2/corpora/{CORPUS_KEY}/query"
query_str = "Are pets allowed in the office?"

payload = json.dumps({
  "query": query_str,
  "search": {
    "lexical_interpolation": 0.025,
    "offset": 0,
    "limit": 50,
    "context_configuration": {
      "sentences_before": 2,
      "sentences_after": 2
    },
    "reranker": {
      "type": "customer_reranker",
      "reranker_name": "Rerank_Multilingual_v1"
    }
  },
  "generation": {
    "max_used_search_results": 7,
    "response_language": "eng",
    "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
    "enable_factual_consistency_score": True
  }
})
headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'x-api-key': VECATARA_API_KEY
}

response = requests.post(url, headers=headers, data=payload)
res = response.json()
search_results = res['search_results']

print(res['summary'])
print(f"Factual Consistency Score: {res['factual_consistency_score']}")

Pets are allowed in the office at Vectara, but with specific guidelines. Birds are permitted and even encouraged in the workspace, although there are particular rules to follow [2]. However, common household pets such as cats and dogs are not allowed on Vectara campuses [7].
Factual Consistency Score: 0.85546875


In [6]:
search_results[:2]

[{'score': 0.9779677391052246,
  'document_metadata': {'Producer': 'Skia/PDF m118 Google Docs Renderer',
   'Title': 'Employee Handbook - Company Pet Policy',
   'title': 'Employee Handbook - Company Pet Policy'},
  'document_id': 'pet_policy',
  'part_metadata': {'page': 3,
   'lang': 'eng',
   'section': 30,
   'offset': 0,
   'len': 89},
  'text': "Strict guidelines, ethical practices, and responsible stewardship are non-negotiable. The Community Outreach: We regularly host educational programs, wildlife awareness\ncampaigns, and community engagement activities. Our exotic pets are not just oﬃce mascots; they are ambassadors of our vision and values. Our exotic pet policy is more than a quirky perk; it's a manifestation of our organizational ethos. It adds a touch of adventure, wisdom, and wildness to our daily grind.",
  'result_type': 'text'},
 {'score': 0.9751545190811157,
  'document_metadata': {'Producer': 'Skia/PDF m118 Google Docs Renderer',
   'Title': 'Employee Handbook - C

### **Hallucination correction**

In [7]:
hallucinated_response = """
Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2]. 
However, common household pets like cats and snakes are not allowed on the Vectara campuses [7].
"""

In [8]:
url = f"https://api.vectara.io/v2/hallucination_correctors/correct_hallucinations"
payload = {
  "generated_text": hallucinated_response,
  "query": query_str,
  "documents": [
      {"text": r["text"]}
      for r in search_results
  ],
  "model_name": "vhc-large-1.0"
}
headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'x-api-key': VECATARA_API_KEY
}

response = requests.post(url, headers=headers, json=payload)
res = response.json()

print("Original (hallucinated):")
print(hallucinated_response)
print("\nCorrected:")
print(res['corrected_text'])

Original (hallucinated):

Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2]. 
However, common household pets like cats and snakes are not allowed on the Vectara campuses [7].


Corrected:
Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are specific guidelines to follow.
However, common household pets like cats and dogs are not allowed on the Vectara campuses.


In [9]:
res['corrections']

[{'original_text': 'Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2].',
  'corrected_text': 'Birds are permitted and even encouraged in the workspace, but there are specific guidelines to follow.',
  'explanation': 'The response incorrectly states that there are "no rules to follow" for bringing birds, while the source clearly indicates there are specific guidelines and peculiarities to this policy.'},
 {'original_text': 'common household pets like cats and snakes are not allowed on the Vectara campuses [7].',
  'corrected_text': 'common household pets like cats and dogs are not allowed on the Vectara campuses.',
  'explanation': "The source specifies that cats and dogs are not allowed, but it does not mention snakes as 'common household pets' or as being not allowed."}]